In [ ]:
from evaluation.args import HeadArgs
from pathlib import Path

import json
import torch
import time

import numpy as np
import tqdm

from models import build_model
from datasets import build_dataset
from util.misc import nested_tensor_from_tensor_list

from evaluation.flops.flop_count import flop_count
from models.backbone import build_backbone
from models.transformer import build_transformer
from torch import nn
from evaluation.flops.compute_flops_breakdown_baseline_detr import (
    get_dataset, 
    measure_time, 
    fmt_res,
    BackboneWrapper,
)
args = HeadArgs(number_of_heads=4)

# get the first 100 images of COCO val2017
PATH_TO_COCO = args.coco_path
dataset = get_dataset(PATH_TO_COCO)
images = []
for idx in range(100):
    img, t = dataset[idx]
    images.append(img)

device = torch.device('cuda')
results = {}
for model_name in ['detr_resnet50_baseline']:
    results[model_name] = []
    with torch.no_grad():
        for head in range(1, args.nheads + 1):
            detr_tmp = []
            backbone_temp = []
            transformer_temp = []
            tmp2 = []
            # Update the number of heads in the args
            args = HeadArgs(number_of_heads=head)
            model = build_model(args)[0].to(device)
            # Rebuild the backbone and transformer with the updated args
            backbone = build_backbone(args).to(device)
            transformer = build_transformer(args).to(device)

            # Compute FLOPS and time for each image
            for img in tqdm.tqdm(images):
                detr_res = flop_count(model, ([img.to(device)],))
                detr_t = measure_time(model, [img.to(device)])
                detr_tmp.append(sum(detr_res.values()))
                tmp2.append(detr_t)

                ## Backbone breakdown
                backbone_nt = nested_tensor_from_tensor_list(img.unsqueeze(0).to(device))
                backbone_wrapper = BackboneWrapper(backbone)
                backbone_res = flop_count(backbone_wrapper, (backbone_nt.tensors, backbone_nt.mask))
                backbone_temp.append(sum(backbone_res.values()))

                ## Transformer breakdown
                input_proj = nn.Conv2d(backbone.num_channels, args.hidden_dim, kernel_size=1).to(device)
                input = nested_tensor_from_tensor_list(img.unsqueeze(0).to(device))
                query_embed = nn.Embedding(args.num_queries, args.hidden_dim).to(device)
                features, pos = backbone(input)
                src, mask = features[-1].decompose()
                transformer_res = flop_count(transformer, (input_proj(src), mask, query_embed.weight, pos[-1]))
                transformer_temp.append(sum(transformer_res.values()))
        
            results[model_name].append({
                'heads': head,
                'detr_flops': fmt_res(np.array(detr_tmp)), 
                'backbone_flops': fmt_res(np.array(backbone_temp)), 
                'transformer_flops': fmt_res(np.array(transformer_temp)), 
                'time': fmt_res(np.array(tmp2))
            })


json_results = {
    model: [
        {k: [float(x) for x in v] if isinstance(v, tuple) else v
        for k, v in entry.items()}
        for entry in entries
    ]
    for model, entries in results.items()
}
output_path = Path('flops_breakdown_results_baseline.json')
with open(output_path, 'w') as f:
    json.dump(json_results, f, indent=2)
print(f'Results saved to {output_path.resolve()}')

In [ ]:
# this is the main entrypoint
# as we describe in the paper, we compute the flops over the first 100 images
# on COCO val2017, and report the average result
from pathlib import Path

import json
import torch
import time

import numpy as np
import tqdm

from sliced_models import build_model
from datasets import build_dataset
from util.misc import nested_tensor_from_tensor_list

from evaluation.flops.flop_count import flop_count
from evaluation.args import HeadArgs
from models.backbone import build_backbone
from sliced_models.transformer.transformer import build_transformer

from torch import nn
from sliced_models.layers.conv import SlicedConv2d

class BackboneWrapper(nn.Module):
    """Wraps the backbone so it accepts plain tensors instead of NestedTensor,
    allowing torch.jit tracing used by flop_count to work."""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, tensors: torch.Tensor, mask: torch.Tensor):
        nt = nested_tensor_from_tensor_list([tensors[i] for i in range(tensors.shape[0])])
        # override the mask with the pre-computed one so shapes stay static
        nt.mask = mask
        features, pos = self.backbone(nt)
        # return the last feature map and its position encoding as plain tensors
        src, src_mask = features[-1].decompose()
        return src, src_mask, pos[-1]
                
def get_dataset(coco_path):
    """
    Gets the COCO dataset used for computing the flops on
    """
    class DummyArgs:
        pass
    args = DummyArgs()
    args.dataset_file = "coco"
    args.coco_path = coco_path
    args.masks = False
    dataset = build_dataset(image_set='val', args=args)
    return dataset


def warmup(model, inputs, N=10):
    for i in range(N):
        if isinstance(inputs, tuple) or isinstance(inputs, list):
            out = model(*inputs)
        elif isinstance(inputs, dict):
            out = model(**inputs)
        else:
            out = model(inputs)
    torch.cuda.synchronize()


def measure_time(model, inputs, N=10):
    warmup(model, inputs)
    s = time.time()
    for i in range(N):
        if isinstance(inputs, tuple) or isinstance(inputs, list):
            out = model(*inputs)
        elif isinstance(inputs, dict):
            out = model(**inputs)
        else:
            out = model(inputs)
    torch.cuda.synchronize()
    t = (time.time() - s) / N
    return t

def fmt_res(data):
    return data.mean(), data.std(), data.min(), data.max()

if __name__ == '__main__':
    args = HeadArgs(number_of_heads=4)

    # get the first 100 images of COCO val2017
    PATH_TO_COCO = args.coco_path
    dataset = get_dataset(PATH_TO_COCO)
    images = []
    for idx in range(100):
        img, t = dataset[idx]
        images.append(img)

    device = torch.device('cuda')
    results = {}
    for model_name in ['sliced_detr_resnet50']:
        results[model_name] = []
        with torch.no_grad():
            for head in range(1, args.nheads + 1):
                detr_tmp = []
                backbone_temp = []
                transformer_temp = []
                tmp2 = []
                # Update the number of heads in the args
                args = HeadArgs(number_of_heads=head)
                model = build_model(args)[0].to(device)
                # Rebuild the backbone and transformer with the updated args
                backbone = build_backbone(args).to(device)
                transformer = build_transformer(args).to(device)

                # Compute FLOPS and time for each image
                for img in tqdm.tqdm(images):
                    detr_res = flop_count(model, ([img.to(device)], head))
                    detr_t = measure_time(model, ([img.to(device)], head))
                    detr_tmp.append(sum(detr_res.values()))
                    tmp2.append(detr_t)

                    ## Backbone breakdown
                    backbone_nt = nested_tensor_from_tensor_list([img.to(device)])
                    backbone_wrapper = BackboneWrapper(backbone)
                    backbone_res = flop_count(backbone_wrapper, (backbone_nt.tensors, backbone_nt.mask))
                    backbone_temp.append(sum(backbone_res.values()))

                    ## Transformer breakdown
                    input_proj = SlicedConv2d(backbone.num_channels, args.hidden_dim, kernel_size=1).to(device)
                    input = nested_tensor_from_tensor_list([img.to(device)])
                    query_embed = nn.Embedding(args.num_queries, args.hidden_dim).to(device)
                    features, pos = backbone(input)
                    src, mask = features[-1].decompose()
                    head_dim = args.hidden_dim // args.nheads
                    d_active = head * head_dim 
                    pos_active = pos[-1][:, :d_active, :, :]
                    transformer_res = flop_count(transformer, (input_proj(src, None, d_out=d_active), mask, query_embed.weight, pos_active, head))
                    transformer_temp.append(sum(transformer_res.values()))
            
                results[model_name].append({
                    'heads': head,
                    'detr_flops': fmt_res(np.array(detr_tmp)), 
                    'backbone_flops': fmt_res(np.array(backbone_temp)), 
                    'transformer_flops': fmt_res(np.array(transformer_temp)), 
                    'time': fmt_res(np.array(tmp2))
                })
                print(results[model_name][-1])


    json_results = {
        model: [
            {k: [float(x) for x in v] if isinstance(v, tuple) else v
            for k, v in entry.items()}
            for entry in entries
        ]
        for model, entries in results.items()
    }
    output_path = Path('flops_breakdown_results_sliced.json')
    with open(output_path, 'w') as f:
        json.dump(json_results, f, indent=2)
    print(f'Results saved to {output_path.resolve()}')

In [ ]:
# this is the main entrypoint
# as we describe in the paper, we compute the flops over the first 100 images
# on COCO val2017, and report the average result
from pathlib import Path

import json
import torch
import time

import numpy as np
import tqdm

from layer_scaling import build_model
from datasets import build_dataset
from util.misc import nested_tensor_from_tensor_list

from evaluation.flops.flop_count import flop_count
from evaluation.args import LayerArgs
from models.backbone import build_backbone
from layer_scaling.transformer import build_transformer

from torch import nn

class BackboneWrapper(nn.Module):
    """Wraps the backbone so it accepts plain tensors instead of NestedTensor,
    allowing torch.jit tracing used by flop_count to work."""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, tensors: torch.Tensor, mask: torch.Tensor):
        nt = nested_tensor_from_tensor_list([tensors[i] for i in range(tensors.shape[0])])
        # override the mask with the pre-computed one so shapes stay static
        nt.mask = mask
        features, pos = self.backbone(nt)
        # return the last feature map and its position encoding as plain tensors
        src, src_mask = features[-1].decompose()
        return src, src_mask, pos[-1]
                
def get_dataset(coco_path):
    """
    Gets the COCO dataset used for computing the flops on
    """
    class DummyArgs:
        pass
    args = DummyArgs()
    args.dataset_file = "coco"
    args.coco_path = coco_path
    args.masks = False
    dataset = build_dataset(image_set='val', args=args)
    return dataset


def warmup(model, inputs, N=10):
    for i in range(N):
        if isinstance(inputs, tuple) or isinstance(inputs, list):
            out = model(*inputs)
        elif isinstance(inputs, dict):
            out = model(**inputs)
        else:
            out = model(inputs)
    torch.cuda.synchronize()


def measure_time(model, inputs, N=10):
    warmup(model, inputs)
    s = time.time()
    for i in range(N):
        if isinstance(inputs, tuple) or isinstance(inputs, list):
            out = model(*inputs)
        elif isinstance(inputs, dict):
            out = model(**inputs)
        else:
            out = model(inputs)
    torch.cuda.synchronize()
    t = (time.time() - s) / N
    return t

def fmt_res(data):
    return data.mean(), data.std(), data.min(), data.max()

if __name__ == '__main__':
    args = LayerArgs(number_of_heads=4, n_layer=6)

    # get the first 100 images of COCO val2017
    PATH_TO_COCO = args.coco_path
    dataset = get_dataset(PATH_TO_COCO)
    images = []
    for idx in range(100):
        img, t = dataset[idx]
        images.append(img)

    device = torch.device('cuda')
    results = {}
    for model_name in ['sliced_detr_resnet50']:
        results[model_name] = []
        with torch.no_grad():
            for head in range(1, args.enc_layers + 1):
                detr_tmp = []
                backbone_temp = []
                transformer_temp = []
                tmp2 = []
                # Update the number of heads in the args
                args = LayerArgs(number_of_heads=4, n_layer=6)
                model = build_model(args)[0].to(device)
                # Rebuild the backbone and transformer with the updated args
                backbone = build_backbone(args).to(device)
                transformer = build_transformer(args).to(device)

                # Compute FLOPS and time for each image
                for img in tqdm.tqdm(images):
                    detr_res = flop_count(model, ([img.to(device)], head))
                    detr_t = measure_time(model, ([img.to(device)], head))
                    detr_tmp.append(sum(detr_res.values()))
                    tmp2.append(detr_t)

                    ## Backbone breakdown
                    backbone_nt = nested_tensor_from_tensor_list([img.to(device)])
                    backbone_wrapper = BackboneWrapper(backbone)
                    backbone_res = flop_count(backbone_wrapper, (backbone_nt.tensors, backbone_nt.mask))
                    backbone_temp.append(sum(backbone_res.values()))

                    ## Transformer breakdown
                    input_proj = nn.Conv2d(backbone.num_channels, args.hidden_dim, kernel_size=1).to(device)
                    input = nested_tensor_from_tensor_list([img.to(device)])
                    query_embed = nn.Embedding(args.num_queries, args.hidden_dim).to(device)
                    features, pos = backbone(input)
                    src, mask = features[-1].decompose()
                    head_dim = args.hidden_dim // args.nheads
                    d_active = head * head_dim 
                    pos_active = pos[-1]
                    transformer_res = flop_count(transformer, (input_proj(src), mask, query_embed.weight, pos_active, head))
                    transformer_temp.append(sum(transformer_res.values()))
            
                results[model_name].append({
                    'heads': head,
                    'detr_flops': fmt_res(np.array(detr_tmp)), 
                    'backbone_flops': fmt_res(np.array(backbone_temp)), 
                    'transformer_flops': fmt_res(np.array(transformer_temp)), 
                    'time': fmt_res(np.array(tmp2))
                })
                print(results[model_name][-1])


    json_results = {
        model: [
            {k: [float(x) for x in v] if isinstance(v, tuple) else v
            for k, v in entry.items()}
            for entry in entries
        ]
        for model, entries in results.items()
    }
    output_path = Path('flops_breakdown_results_layer_scaled.json')
    with open(output_path, 'w') as f:
        json.dump(json_results, f, indent=2)
    print(f'Results saved to {output_path.resolve()}')

In [ ]:

import json
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pathlib import Path

RESULTS_DIR = Path(r'C:\workspace\ml\workspace\master\original\results')
MEAN = 0

def parse_flops_series(data, model_key, config_key='heads'):
    seen, entries = set(), []
    for e in data[model_key]:
        h = e[config_key]
        if h not in seen:
            seen.add(h)
            entries.append(e)
    entries.sort(key=lambda x: x[config_key])
    return [e[config_key] for e in entries], [e['detr_flops'][MEAN] for e in entries]


with open(RESULTS_DIR / 'baseline' / 'baseline_flops_breakdown_results.json') as f:
    orig_data = json.load(f)
with open(RESULTS_DIR / 'sliced' / 'flops_breakdown_results_sliced.json') as f:
    sliced_data = json.load(f)
with open(RESULTS_DIR / 'layer_scaling' / 'flops_breakdown_results_layer_scaled.json') as f:
    layer_scaling_data = json.load(f)

orig_heads, orig_flops = parse_flops_series(orig_data, 'detr_resnet50_baseline')
sliced_heads, sliced_flops = parse_flops_series(sliced_data, 'sliced_detr_resnet50')
layer_scaling_layers, layer_scaling_flops = parse_flops_series(layer_scaling_data, 'layer_scaled_detr_resnet50', config_key='heads') # In the cell above it uses config_key = heads for layer scaling

C_ORIG = '#1B4F72'
C_SLICED = '#C0392B'
C_LAYER = '#8E44AD'

plt.style.use('seaborn-v0_8-poster')
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(orig_heads, orig_flops, color=C_ORIG, linewidth=2.5, linestyle='-.',
        marker='o', markersize=9, markerfacecolor='white', markeredgewidth=2, zorder=4)
ax.plot(sliced_heads, sliced_flops, color=C_SLICED, linewidth=2.5, linestyle='--',
        marker='s', markersize=9, markerfacecolor='white', markeredgewidth=2, zorder=3)
ax.plot(layer_scaling_layers, layer_scaling_flops, color=C_LAYER, linewidth=2.5, linestyle=':',
        marker='D', markersize=9, markerfacecolor='white', markeredgewidth=2, zorder=3)

ax.axhline(y=75, color='gray', linestyle='--', alpha=0.5, zorder=1)
ax.axhline(y=76, color='gray', linestyle='--', alpha=0.5, zorder=1)
ax.axhline(y=77, color='gray', linestyle='--', alpha=0.5, zorder=1)

ax.text(x=0.9, y=75.1, s='75', color='gray', alpha=0.7, fontsize=10, zorder=1)
ax.text(x=0.9, y=76.1, s='76', color='gray', alpha=0.7, fontsize=10, zorder=1)
ax.text(x=0.9, y=77.1, s='77', color='gray', alpha=0.7, fontsize=10, zorder=1)


legend_handles = [
    mlines.Line2D([], [], color=C_ORIG,   linestyle='-',  marker='o', linewidth=2.5, markersize=8,
                  markerfacecolor='white', label='Baseline'),
    mlines.Line2D([], [], color=C_SLICED, linestyle='--', marker='s', linewidth=2.5, markersize=8,
                  markerfacecolor='white', label='Width-Scalable'),
    mlines.Line2D([], [], color=C_LAYER,  linestyle=':',  marker='D', linewidth=2.5, markersize=8,
                  markerfacecolor='white', label='Depth-Scalable'),
]
ax.legend(handles=legend_handles, fontsize=11, frameon=True)

ax.set_xlabel('Configuration', fontsize=13, fontweight='bold')
ax.set_ylabel('Total MACs [G] (mean)', fontsize=13, fontweight='bold')
#ax.set_title('DETR-ResNet50: Total GMACs vs Configuration', fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(sorted(set(orig_heads + sliced_heads + layer_scaling_layers)))
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'detr_flops_per_configuration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'Config':>5} | {'Original':>12} | {'Sliced':>12} | {'Layer-Scaling':>13} | {'Δ Sliced':>10} | {'Δ Layer':>10} | {'Step Sliced':>13} | {'Step Layer':>13}")
print("-" * 115)
orig_map = dict(zip(orig_heads, orig_flops))
sliced_map = dict(zip(sliced_heads, sliced_flops))
layer_scaling_map = dict(zip(layer_scaling_layers, layer_scaling_flops))

prev_s = None
prev_ls = None

for h in sorted(set(orig_heads) | set(sliced_heads) | set(layer_scaling_layers)):
    o = orig_map.get(h)
    s = sliced_map.get(h)
    ls = layer_scaling_map.get(h)
    
    ds = f'{s - o:+.4f}' if o is not None and s is not None else '-'
    dls = f'{ls - o:+.4f}' if o is not None and ls is not None else '-'
    
    step_s = f'{s - prev_s:+.4f}' if s is not None and prev_s is not None else '-'
    step_ls = f'{ls - prev_ls:+.4f}' if ls is not None and prev_ls is not None else '-'
    
    o_str = f'{o:12.4f}' if o is not None else f"{'−':>12}"
    s_str = f'{s:12.4f}' if s is not None else f"{'−':>12}"
    ls_str = f'{ls:12.4f}' if ls is not None else f"{'−':>12}"
    
    print(f"{h:>5} | {o_str} | {s_str} | {ls_str:>13} | {ds:>10} | {dls:>10} | {step_s:>13} | {step_ls:>13}")
    
    if s is not None: prev_s = s
    if ls is not None: prev_ls = ls


In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

RESULTS_DIR = Path(r'C:\workspace\ml\workspace\master\original\results')
MEAN_IDX = 0

# Load all three data files
with open(RESULTS_DIR / 'baseline' / 'baseline_flops_breakdown_results.json') as f:
    baseline_data = json.load(f)

with open(RESULTS_DIR / 'sliced' / 'flops_breakdown_results_sliced.json') as f:
    sliced_data = json.load(f)

with open(RESULTS_DIR / 'layer_scaling' / 'flops_breakdown_results_layer_scaled.json') as f:
    layer_scaling_data = json.load(f)

def extract_components(data, model_key):
    """Extract heads and component arrays from data"""
    heads = []
    backbone = []
    transformer = []
    other = []
    
    for entry in data[model_key]:
        h = entry['heads']
        heads.append(h)
        
        b = entry['backbone_flops'][MEAN_IDX]
        t = entry['transformer_flops'][MEAN_IDX]
        d = entry['detr_flops'][MEAN_IDX]
        o = d - b - t
        
        backbone.append(b)
        transformer.append(t)
        other.append(o)
    
    return np.array(heads), np.array(backbone), np.array(transformer), np.array(other)

# Extract data for all models
heads_b, backbone_b, transformer_b, other_b = extract_components(baseline_data, 'detr_resnet50_baseline')
heads_s, backbone_s, transformer_s, other_s = extract_components(sliced_data, 'sliced_detr_resnet50')
heads_ls, backbone_ls, transformer_ls, other_ls = extract_components(layer_scaling_data, 'layer_scaled_detr_resnet50')

# Define colors for each model variant
colors_baseline = {'backbone': '#1B4F72', 'transformer': '#FF0000', 'other': '#27AE60'}
colors_sliced = {'backbone': '#2E86C1', 'transformer': '#FF0000', 'other': '#52BE80'}
colors_layer = {'backbone': '#5DADE2', 'transformer': '#FF0000', 'other': '#76D7C4'}

linestyles = {'baseline': '-', 'sliced': '--', 'layer': ':'}
markers = {'baseline': 'o', 'sliced': 's', 'layer': '^'}

# Figure 1: Baseline
fig1, ax = plt.subplots(figsize=(10, 6))
ax.set_facecolor('#F0F0F0')
# Plot only transformer line
ax.plot(heads_b, backbone_b + transformer_b, color=colors_baseline['transformer'], linestyle=linestyles['baseline'],
        marker=markers['baseline'], linewidth=3.5, markersize=8, label='Transformer', zorder=3)
# Fill areas
ax.fill_between(heads_b, 70, backbone_b, alpha=0.15, color=colors_baseline['backbone'], zorder=1)
ax.fill_between(heads_b, backbone_b, backbone_b + transformer_b, alpha=0.15, color=colors_baseline['transformer'], zorder=1)
ax.fill_between(heads_b, backbone_b + transformer_b, backbone_b + transformer_b + other_b, alpha=0.15, color=colors_baseline['other'], zorder=1)

# Create legend with patches
legend_elements = [
    mpatches.Patch(facecolor=colors_baseline['backbone'], alpha=0.15, edgecolor=colors_baseline['backbone'], label='Backbone'),
    mpatches.Patch(facecolor=colors_baseline['transformer'], alpha=0.15, edgecolor=colors_baseline['transformer'], label='Transformer'),
    mpatches.Patch(facecolor=colors_baseline['other'], alpha=0.15, edgecolor=colors_baseline['other'], label='Other'),
]
ax.legend(handles=legend_elements, fontsize=11, loc='upper left')

#ax.set_title('Baseline DETR-ResNet50', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Configuration (# Heads)', fontsize=12, fontweight='bold')
ax.set_ylabel('MACs [G] (mean)', fontsize=12, fontweight='bold')
ax.set_ylim(70, None)
ax.grid(True, alpha=0.3, linestyle='--', zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks(heads_b)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'detr_flops_breakdown_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 2: Sliced
fig2, ax = plt.subplots(figsize=(10, 6))
ax.set_facecolor('#F0F0F0')
# Plot only transformer line
ax.plot(heads_s, backbone_s + transformer_s, color=colors_sliced['transformer'], linestyle=linestyles['sliced'],
        marker=markers['sliced'], linewidth=3.5, markersize=8, label='Transformer', zorder=3)
# Fill areas
ax.fill_between(heads_s, 70, backbone_s, alpha=0.15, color=colors_sliced['backbone'], zorder=1)
ax.fill_between(heads_s, backbone_s, backbone_s + transformer_s, alpha=0.15, color=colors_sliced['transformer'], zorder=1)
ax.fill_between(heads_s, backbone_s + transformer_s, backbone_s + transformer_s + other_s, alpha=0.15, color=colors_sliced['other'], zorder=1)

# Create legend with patches
legend_elements = [
    mpatches.Patch(facecolor=colors_sliced['backbone'], alpha=0.15, edgecolor=colors_sliced['backbone'], label='Backbone'),
    mpatches.Patch(facecolor=colors_sliced['transformer'], alpha=0.15, edgecolor=colors_sliced['transformer'], label='Transformer'),
    mpatches.Patch(facecolor=colors_sliced['other'], alpha=0.15, edgecolor=colors_sliced['other'], label='Other'),
]
ax.legend(handles=legend_elements, fontsize=11, loc='upper left')

#ax.set_title('Width-Sliced DETR-ResNet50', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Configuration (# Heads)', fontsize=12, fontweight='bold')
ax.set_ylabel('MACs [G] (mean)', fontsize=12, fontweight='bold')
ax.set_ylim(70, None)
ax.grid(True, alpha=0.3, linestyle='--', zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks(heads_s)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'detr_flops_breakdown_sliced.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 3: Layer Scaling
fig3, ax = plt.subplots(figsize=(10, 6))
ax.set_facecolor('#F0F0F0')
# Plot only transformer line
ax.plot(heads_ls, backbone_ls + transformer_ls, color=colors_layer['transformer'], linestyle=linestyles['layer'],
        marker=markers['layer'], linewidth=3.5, markersize=8, label='Transformer', zorder=3)
# Fill areas
ax.fill_between(heads_ls, 70, backbone_ls, alpha=0.15, color=colors_layer['backbone'], zorder=1)
ax.fill_between(heads_ls, backbone_ls, backbone_ls + transformer_ls, alpha=0.15, color=colors_layer['transformer'], zorder=1)
ax.fill_between(heads_ls, backbone_ls + transformer_ls, backbone_ls + transformer_ls + other_ls, alpha=0.15, color=colors_layer['other'], zorder=1)

# Create legend with patches
legend_elements = [
    mpatches.Patch(facecolor=colors_layer['backbone'], alpha=0.15, edgecolor=colors_layer['backbone'], label='Backbone'),
    mpatches.Patch(facecolor=colors_layer['transformer'], alpha=0.15, edgecolor=colors_layer['transformer'], label='Transformer'),
    mpatches.Patch(facecolor=colors_layer['other'], alpha=0.15, edgecolor=colors_layer['other'], label='Other'),
]
ax.legend(handles=legend_elements, fontsize=11, loc='upper left')

#ax.set_title('Depth-Scaled DETR-ResNet50', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Configuration (# Layers)', fontsize=12, fontweight='bold')
ax.set_ylabel('MACs [G] (mean)', fontsize=12, fontweight='bold')
ax.set_ylim(70, None)
ax.grid(True, alpha=0.3, linestyle='--', zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks(heads_ls)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'detr_flops_breakdown_layer_scaled.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*120)
print("DETR-ResNet50: FLOPs Component Breakdown Comparison")
print("="*120)
print(f"\n{'Config':<8} | {'Model':<15} | {'Backbone':>12} | {'Backbone %':>10} | {'Transformer':>12} | {'Trans %':>8} | {'Other':>10} | {'Other %':>8} | {'Total':>10}")
print("-"*120)

for heads_val in sorted(set(heads_b) | set(heads_s) | set(heads_ls)):
    for model_name, heads, backbone, transformer, other in [
        ('Baseline', heads_b, backbone_b, transformer_b, other_b),
        ('Sliced', heads_s, backbone_s, transformer_s, other_s),
        ('Layer-Scaled', heads_ls, backbone_ls, transformer_ls, other_ls),
    ]:
        idx = np.where(heads == heads_val)[0]
        if len(idx) > 0:
            b_val = backbone[idx[0]]
            t_val = transformer[idx[0]]
            o_val = other[idx[0]]
            total = b_val + t_val + o_val
            b_pct = (b_val / total * 100) if total > 0 else 0
            t_pct = (t_val / total * 100) if total > 0 else 0
            o_pct = (o_val / total * 100) if total > 0 else 0
            print(f"{heads_val:<8} | {model_name:<15} | {b_val:>12.2f} | {b_pct:>9.1f}% | {t_val:>12.2f} | {t_pct:>7.1f}% | {o_val:>10.2f} | {o_pct:>7.1f}% | {total:>10.2f}")
    print("-"*120)
